In [1]:
import numpy as np
import pandas as pd

df=pd.read_excel('../data/raw/Initial_Public_Offering.xlsx')
print("Shape befor Drop: ",df.shape)
unnamed_cols=[col for col in df.columns if 'Unnamed' in str(col)]
df=df.drop(columns=unnamed_cols)

print("shape after Drop: ",df.shape)
print("Columns Reamaining: ",df.columns.tolist())

Shape befor Drop:  (561, 21)
shape after Drop:  (561, 13)
Columns Reamaining:  ['Date', 'IPO_Name', 'Issue_Size(crores)', 'QIB', 'HNI', 'RII', 'Total', 'Offer Price', 'List Price', 'Listing Gain', 'CMP(BSE)', 'CMP(NSE)', 'Current Gains']


In [2]:
print(df.head())
print("********************************************************************")
print(df.info())
print("********************************************************************")
print(df.describe())

        Date                                   IPO_Name  Issue_Size(crores)  \
0 2025-08-06                      M & B Engineering Ltd              650.00   
1 2025-08-06          Sri Lotus Developers & Realty Ltd              792.00   
2 2025-08-06  National Securities Depository Ltd (NSDL)             4011.60   
3 2025-08-05                        Aditya Infotech Ltd             1300.00   
4 2025-08-05                    Laxmi India Finance Ltd              254.26   

      QIB    HNI    RII   Total  Offer Price  List Price  Listing Gain  \
0   36.72  38.24  32.55   36.20          385       386.0          0.26   
1  163.90  57.71  20.28   69.14          150       179.1         19.40   
2  103.97  34.98   7.73   41.01          800       880.0         10.00   
3  133.21  72.00  50.87  100.69          675      1018.0         50.81   
4    1.30   1.84   2.22    1.87          158       136.0        -13.92   

   CMP(BSE)  CMP(NSE)  Current Gains  
0    426.85    426.15          10.87  
1 

## Null Check + Duplicate Check

In [3]:
null_counts=df.isnull().sum().sort_values(ascending=False)
print("Null Count Per Column")
print(null_counts)

null_pct=(df.isnull().sum()/len(df)*100).round(2).sort_values(ascending=False)
print("\n Null % Per Column")
print(null_pct)

Null Count Per Column
CMP(NSE)              10
Current Gains          3
QIB                    2
HNI                    2
RII                    2
Total                  2
CMP(BSE)               2
Date                   0
IPO_Name               0
Issue_Size(crores)     0
Offer Price            0
List Price             0
Listing Gain           0
dtype: int64

 Null % Per Column
CMP(NSE)              1.78
Current Gains         0.53
QIB                   0.36
HNI                   0.36
RII                   0.36
Total                 0.36
CMP(BSE)              0.36
Date                  0.00
IPO_Name              0.00
Issue_Size(crores)    0.00
Offer Price           0.00
List Price            0.00
Listing Gain          0.00
dtype: float64


In [4]:
total_dupes=df.duplicated().sum()
print(f"Total duplicate rows: {total_dupes}")

name_dupes=df.duplicated(subset=['IPO_Name','Date']).sum()
print(f"Duplicate IPO_Name + date combination: {name_dupes}")

if name_dupes>0:
    print(df[df.duplicated(
        subset=['IPO_Name','Date'],keep=False)])

Total duplicate rows: 0
Duplicate IPO_Name + date combination: 0


In [5]:
print("Data Types")
print(df.dtypes)

Data Types
Date                  datetime64[ns]
IPO_Name                      object
Issue_Size(crores)           float64
QIB                          float64
HNI                          float64
RII                          float64
Total                        float64
Offer Price                    int64
List Price                   float64
Listing Gain                 float64
CMP(BSE)                     float64
CMP(NSE)                     float64
Current Gains                float64
dtype: object


## Null Diagnosis — Final Verified

**QIB, HNI, RII, Total — rows 297 and 411**

- Mindspace Business Parks REIT (2020): A REIT is a Real 
  Estate Investment Trust — it is NOT a regular IPO. REITs 
  follow different subscription rules and do not always 
  report QIB/HNI/RII splits the same way. Missing data 
  is expected.

- Coffee Day Enterprises Ltd (2015): Subscription data 
  was simply not captured at source for this IPO.

**CMP(BSE) — rows 367 and 379**

- Central Depository Services Ltd (CDSL): Has CMP(NSE) = 
  1556.0 but CMP(BSE) is null. This is a data entry issue 
  at source — the stock actively trades on both exchanges. 
  BSE value was not recorded.

- BSE Ltd (Bombay Stock Exchange): The stock exchange 
  itself is listed as a company. Same issue — NSE price 
  exists (2363.6) but BSE price missing at source.

**Key insight: CMP(BSE) nulls are NOT delisted companies.**
Both CDSL and BSE Ltd are actively trading stocks. 
Their CMP(NSE) values exist. This means we can FILL 
CMP(BSE) nulls using CMP(NSE) values in Day 5 — 
instead of dropping these rows entirely.

## Fix Data Types

In [6]:
print("Data types")
print(df.dtypes)

Data types
Date                  datetime64[ns]
IPO_Name                      object
Issue_Size(crores)           float64
QIB                          float64
HNI                          float64
RII                          float64
Total                        float64
Offer Price                    int64
List Price                   float64
Listing Gain                 float64
CMP(BSE)                     float64
CMP(NSE)                     float64
Current Gains                float64
dtype: object


In [7]:
print("Sample Values per column")
for col in df.columns:
    print(f"(col): {df[col].iloc[0]} | type: {type(df[col].iloc[0])}")

Sample Values per column
(col): 2025-08-06 00:00:00 | type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>
(col): M & B Engineering Ltd | type: <class 'str'>
(col): 650.0 | type: <class 'numpy.float64'>
(col): 36.72 | type: <class 'numpy.float64'>
(col): 38.24 | type: <class 'numpy.float64'>
(col): 32.55 | type: <class 'numpy.float64'>
(col): 36.2 | type: <class 'numpy.float64'>
(col): 385 | type: <class 'numpy.int64'>
(col): 386.0 | type: <class 'numpy.float64'>
(col): 0.26 | type: <class 'numpy.float64'>
(col): 426.85 | type: <class 'numpy.float64'>
(col): 426.15 | type: <class 'numpy.float64'>
(col): 10.87 | type: <class 'numpy.float64'>


In [8]:
numeric_cols=['Issue_Size(crores)', 'QIB', 'HNI', 'RII', 'Total',
                'Offer Price', 'List Price', 'Listing Gain',
                'CMP(BSE)', 'CMP(NSE)', 'Current Gains']
print("checking for text in columns")
for col in numeric_cols:
    try:
        pd.to_numeric(df[col],errors='raise')
        print(f"{col} - Clean No hidden chars")
    except:
        print(f"{col} - Warning conatains non-numeric values")

checking for text in columns
Issue_Size(crores) - Clean No hidden chars
QIB - Clean No hidden chars
HNI - Clean No hidden chars
RII - Clean No hidden chars
Total - Clean No hidden chars
Offer Price - Clean No hidden chars
List Price - Clean No hidden chars
Listing Gain - Clean No hidden chars
CMP(BSE) - Clean No hidden chars
CMP(NSE) - Clean No hidden chars
Current Gains - Clean No hidden chars


In [9]:
df['Offer Price']=df['Offer Price'].astype(float)
df['Date']=pd.to_datetime(df['Date'])
df['IPO_Name']=df['IPO_Name'].astype(str)
print("After Clean: Data Types")
print(df.dtypes)
print("Null check after type fix")
print(df.isnull().sum())

After Clean: Data Types
Date                  datetime64[ns]
IPO_Name                      object
Issue_Size(crores)           float64
QIB                          float64
HNI                          float64
RII                          float64
Total                        float64
Offer Price                  float64
List Price                   float64
Listing Gain                 float64
CMP(BSE)                     float64
CMP(NSE)                     float64
Current Gains                float64
dtype: object
Null check after type fix
Date                   0
IPO_Name               0
Issue_Size(crores)     0
QIB                    2
HNI                    2
RII                    2
Total                  2
Offer Price            0
List Price             0
Listing Gain           0
CMP(BSE)               2
CMP(NSE)              10
Current Gains          3
dtype: int64


## Handle Nulls

In [10]:
print("Shape before dropping QIB nulls:",df.shape)
df=df.dropna(subset=['QIB'])
print("Shape after dropping QIB Nulls:",df.shape)
print("QIB nulls remaining:",df['QIB'].isnull().sum())
print("HNI nulls remaining:",df['HNI'].isnull().sum())
print("RII nulls remaining:",df['RII'].isnull().sum())
print("Total nulls reamaining:",df['Total'].isnull().sum())


Shape before dropping QIB nulls: (561, 13)
Shape after dropping QIB Nulls: (559, 13)
QIB nulls remaining: 0
HNI nulls remaining: 0
RII nulls remaining: 0
Total nulls reamaining: 0


In [11]:
# Fill CMP(BSE) nulls using CMP(NSE) where BSE is missing
df['CMP(BSE)'] = df['CMP(BSE)'].fillna(df['CMP(NSE)'])

# Verify
print("CMP(BSE) nulls remaining:", df['CMP(BSE)'].isnull().sum())
print(df.loc[df['IPO_Name'].isin(['Central Depository Services Ltd',
             'BSE Ltd (Bombay Stock Exchange)']),
             ['IPO_Name', 'CMP(BSE)', 'CMP(NSE)']])

CMP(BSE) nulls remaining: 0
                            IPO_Name  CMP(BSE)  CMP(NSE)
367  Central Depository Services Ltd    1556.0    1556.0
379  BSE Ltd (Bombay Stock Exchange)    2363.6    2363.6


In [12]:
df = df.dropna(subset=['CMP(NSE)'])
print("shape after dropping CMP(NSE) nulls:", df.shape)
print("CMP(NSE) nulls remaining:", df['CMP(NSE)'].isnull().sum())

shape after dropping CMP(NSE) nulls: (549, 13)
CMP(NSE) nulls remaining: 0


In [13]:
print("Current Gains nulls before fix:", df['Current Gains'].isnull().sum())
# Recalculate Current Gains where it is null using BSE price
df['Current Gains'] = df.apply(
    lambda row: ((row['CMP(BSE)'] - row['Offer Price']) / row['Offer Price'] * 100)
    if pd.isnull(row['Current Gains']) else row['Current Gains'],
    axis=1
)
print("Current Gains nulls after fix:", df['Current Gains'].isnull().sum())

Current Gains nulls before fix: 3
Current Gains nulls after fix: 0


In [14]:
print("FINAL NULL CHECK")
print(df.isnull().sum())

print("\nFINAL SHAPE")
print(df.shape)

FINAL NULL CHECK
Date                  0
IPO_Name              0
Issue_Size(crores)    0
QIB                   0
HNI                   0
RII                   0
Total                 0
Offer Price           0
List Price            0
Listing Gain          0
CMP(BSE)              0
CMP(NSE)              0
Current Gains         0
dtype: int64

FINAL SHAPE
(549, 13)


## Null Handling Summary

Started with 561 rows. Ended with 549 rows.
12 rows removed — only 2.1% of data lost.

- 2 rows dropped: Mindspace REIT + Coffee Day — 
  missing subscription data, core of our analysis
- 10 rows dropped: BSE-only companies — 
  no NSE price, cannot compare across exchanges
- 2 CMP(BSE) nulls filled using CMP(NSE) — 
  price parity between exchanges makes this valid
- 3 Current Gains nulls recalculated from formula —
  no data loss, just recomputed from existing columns

Dataset is now 100% null-free and ready for 
feature engineering.

In [17]:
df['Year']=df['Date'].dt.year
df['Month']=df['Date'].dt.month
print(df[['Date','Month','Year']].head())

        Date  Month  Year
0 2025-08-06      8  2025
1 2025-08-06      8  2025
2 2025-08-06      8  2025
3 2025-08-05      8  2025
4 2025-08-05      8  2025


In [22]:
listing_bins  = [-float('inf'), -5, 0, 20, float('inf')]
listing_labels = ['Loss', 'Flat', 'Moderate Gain', 'Strong Gain']
df['Listing_Category']=pd.cut(df['Listing Gain'],bins=listing_bins,labels=listing_labels)
print(df[['Listing Gain','Listing_Category']].head(),df['Listing_Category'].value_counts())

   Listing Gain Listing Category
0          0.26    Moderate Gain
1         19.40    Moderate Gain
2         10.00    Moderate Gain
3         50.81      Strong Gain
4        -13.92             Loss Listing Category
Moderate Gain    209
Strong Gain      172
Flat              96
Loss              72
Name: count, dtype: int64


In [24]:
size_bins   = [-float('inf'), 500, 2000, float('inf')]
size_labels = ['Small', 'Medium', 'Large']
df['Issue_Size_Category'] = pd.cut(df['Issue_Size(crores)'],bins=size_bins,labels=size_labels)
print(df['Issue_Size_Category'].value_counts())

Issue_Size_Category
Small     231
Medium    228
Large      90
Name: count, dtype: int64


In [27]:
sub_bins   = [-float('inf'), 1, 10, 50, float('inf')]
sub_labels = ['Under', 'Low', 'Medium', 'High']
df['Subscription_Tier'] = pd.cut(df['Total'],bins=sub_bins,labels=sub_labels)
print(df['Subscription_Tier'].value_counts())

Subscription_Tier
Low       244
High      151
Medium    133
Under      21
Name: count, dtype: int64


In [28]:
df['QIB_vs_RII'] = df['QIB'] - df['RII']
print(df['QIB_vs_RII'].describe())
inst_dominant = (df['QIB_vs_RII'] > 0).sum()
retail_dominant = (df['QIB_vs_RII'] < 0).sum()
print(f"\nIPOs where institutions > retail : {inst_dominant}")
print(f"IPOs where retail > institutions: {retail_dominant}")

count    549.000000
mean      32.166995
std       57.506445
min     -331.970000
25%        0.520000
50%        7.000000
75%       50.300000
max      266.610000
Name: QIB_vs_RII, dtype: float64

IPOs where institutions > retail : 430
IPOs where retail > institutions: 118


In [29]:
print("NEW COLUMNS ADDED")
print(df.columns.tolist())
print("\nFINAL SHAPE")
print(df.shape)
df.to_csv('../data/cleaned/ipo_cleaned.csv', index=False)
print("\nFile saved to data/cleaned/ipo_cleaned.csv")

NEW COLUMNS ADDED
['Date', 'IPO_Name', 'Issue_Size(crores)', 'QIB', 'HNI', 'RII', 'Total', 'Offer Price', 'List Price', 'Listing Gain', 'CMP(BSE)', 'CMP(NSE)', 'Current Gains', 'Year', 'Month', 'Listing Category', 'Issue_Size_Category', 'Subscription_Tier', 'QIB_vs_RII']

FINAL SHAPE
(549, 19)

File saved to data/cleaned/ipo_cleaned.csv
